In [1]:
import pandas as pd
import numpy as np

from geopy.distance import geodesic

In [2]:
df = pd.read_csv(
    "../data/featured/feature_engineered_data.csv"
)

In [3]:
df[
    ['Factory','Latitude','Longitude']
].head()

,Factory,Latitude,Longitude
0,Wicked Choccy's,32.076176,-81.088371
1,Wicked Choccy's,32.076176,-81.088371
2,Lot's O' Nuts,32.881893,-111.768036
3,Lot's O' Nuts,32.881893,-111.768036
4,Wicked Choccy's,32.076176,-81.088371


In [4]:
region_coordinates = {
    'Atlantic': (40.7128, -74.0060),
    'Gulf': (29.7604, -95.3698),
    'Interior': (39.0997, -94.5786),
    'Pacific': (34.0522, -118.2437)
}

In [5]:
df['Region_Latitude'] = (
    df['Region']
    .map(
        lambda x:
        region_coordinates[x][0]
    )
)

df['Region_Longitude'] = (
    df['Region']
    .map(
        lambda x:
        region_coordinates[x][1]
    )
)

In [6]:
def calculate_distance(row):

    factory = (
        row['Latitude'],
        row['Longitude']
    )

    destination = (
        row['Region_Latitude'],
        row['Region_Longitude']
    )

    return geodesic(
        factory,
        destination
    ).kilometers

In [7]:
df['Shipping_Distance_KM'] = (
    df.apply(
        calculate_distance,
        axis=1
    )
)

In [8]:
df[
[
'Factory',
'Region',
'Shipping_Distance_KM'
]
].head()

,Factory,Region,Shipping_Distance_KM
0,Wicked Choccy's,Interior,1447.402305
1,Wicked Choccy's,Interior,1447.402305
2,Lot's O' Nuts,Interior,1693.005751
3,Lot's O' Nuts,Interior,1693.005751
4,Wicked Choccy's,Atlantic,1148.912517


In [9]:
df['Distance_Category'] = pd.cut(

    df['Shipping_Distance_KM'],

    bins=[
        0,
        1000,
        2000,
        3000,
        5000
    ],

    labels=[
        'Short',
        'Medium',
        'Long',
        'Very Long'
    ]
)

In [10]:
max_distance = (
    df['Shipping_Distance_KM']
    .max()
)

In [11]:
df['Distance_Efficiency'] = (

    1 -

    (
        df['Shipping_Distance_KM']
        /
        max_distance
    )

)

In [13]:
factory_distance.columns = [

    'Factory',

    'Factory_Avg_Distance'

]

In [14]:
df = df.merge(
    factory_distance,
    on='Factory'
)

In [15]:
df['Route_Risk_Score'] = (

    df['Shipping_Distance_KM']
    *
    df['Lead_Time']

)

In [16]:
df['Distance_Profit_Ratio'] = (

    df['Gross Profit']
    /
    (
        df['Shipping_Distance_KM']
        + 1
    )

)

In [18]:
df['Route_Performance_Index'] = (

    df['Profit_Margin']

    *
    df['Distance_Efficiency']

)

In [19]:
distance_features = [

    'Shipping_Distance_KM',

    'Distance_Efficiency',

    'Factory_Avg_Distance',

    'Route_Risk_Score',

    'Distance_Profit_Ratio',

    'Route_Performance_Index'

]

In [20]:
df[
    distance_features
].describe()

,Shipping_Distance_KM,Distance_Efficiency,Factory_Avg_Distance,Route_Risk_Score,Distance_Profit_Ratio,Route_Performance_Index
count,10194.000000,10194.000000,10194.000000,1.019400e+04,10194.000000,10194.000000
mean,1901.268591,0.450136,1901.268591,2.516329e+06,0.006893,0.299039
std,1069.216369,0.309227,97.324341,1.533572e+06,0.008152,0.210505
min,429.335657,0.000000,1527.329837,3.885488e+05,0.000097,0.000000
25%,1148.912517,0.001719,1847.946642,1.450010e+06,0.002833,0.001226
50%,1596.927113,0.538154,1847.946642,2.037679e+06,0.004427,0.373718
75%,3451.764387,0.667724,2001.617854,3.146514e+06,0.008108,0.433507
max,3457.708101,0.875832,2001.617854,5.677557e+06,0.255614,0.700666


In [21]:


df.to_csv(
    "../data/model_ready/model_ready_data.csv",
    index=False
)

# Deliverables 